# Day 4 — Mini Project: Email Spam Detection
## 30 Days of AI: From NLP to LLMs

---

Everything covered in Days 1 to 3 — text cleaning, tokenization,
stopwords, TF-IDF — comes together today in one real end-to-end project.
Spam detection is one of the most widely deployed NLP systems in the
world. By the end of this notebook you will have a fully working
classifier that can predict whether any new email is spam or ham.

---

### What You Will Build

- Clean and preprocess raw email text
- Convert text into TF-IDF feature vectors
- Train a Logistic Regression classifier
- Evaluate with confusion matrix and classification report
- Identify the top words the model learned as spam signals
- Wrap everything into a reusable prediction function

---

### Why TF-IDF Works Well for Spam

Spam emails consistently use words like `free`, `winner`, `urgent`,
`click here`, `limited time`. These words are rare in legitimate email
but very frequent in spam. TF-IDF naturally gives high weight to words
that are rare across the corpus but frequent in a specific document —
exactly the pattern that separates spam from ham.

---

In [1]:
# ── Email Spam Detection ──────────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
import string

# ── Sample Dataset ────────────────────────────────────────────────────
emails = [
    ("Win a FREE iPhone now! Click here to claim your prize!!!", "spam"),
    ("Congratulations! You have been selected for a $1000 gift card", "spam"),
    ("URGENT: Your bank account has been compromised. Verify now", "spam"),
    ("Get rich quick! Make $5000 from home with no experience", "spam"),
    ("Buy cheap viagra online. No prescription needed!!!", "spam"),
    ("You are a WINNER! Claim your free vacation package today", "spam"),
    ("Limited time offer! 90% discount on all products. Act NOW", "spam"),
    ("Dear user, your password will expire. Click to reset immediately", "spam"),
    ("Meet singles in your area tonight. Free registration!!!", "spam"),
    ("Earn money fast! Work from home opportunity. Apply now", "spam"),
    ("Hey, are we still on for lunch tomorrow at 1pm?", "ham"),
    ("Please find the meeting notes attached from today's standup", "ham"),
    ("Your order has been shipped and will arrive by Friday", "ham"),
    ("Can you review the pull request I submitted this morning?", "ham"),
    ("Reminder: team retrospective is scheduled for Thursday 3pm", "ham"),
    ("Hi, I wanted to follow up on the proposal we discussed", "ham"),
    ("The quarterly report is ready for your review", "ham"),
    ("Your appointment is confirmed for Monday at 10am", "ham"),
    ("Thanks for your help with the project last week", "ham"),
    ("Could you send me the updated budget spreadsheet?", "ham"),
]

df = pd.DataFrame(emails, columns=["text", "label"])
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})


# ── Text Cleaning ─────────────────────────────────────────────────────
def clean_email(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)       # remove URLs
    text = re.sub(r'\d+', '', text)                     # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_email)


# ── Train / Test Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label_num"],
    test_size=0.2, random_state=42, stratify=df["label_num"]
)


# ── TF-IDF Vectorization ──────────────────────────────────────────────
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)


# ── Train Model ───────────────────────────────────────────────────────
model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)


# ── Evaluate ──────────────────────────────────────────────────────────
y_pred = model.predict(X_test_vec)

print("Model Evaluation")
print("=" * 45)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

print("Confusion Matrix")
print("-" * 25)
cm = confusion_matrix(y_test, y_pred)
print(f"             Predicted Ham  Predicted Spam")
print(f"Actual Ham       {cm[0][0]:<10}     {cm[0][1]}")
print(f"Actual Spam      {cm[1][0]:<10}     {cm[1][1]}")


# ── Top Spam Indicators ───────────────────────────────────────────────
feature_names = vectorizer.get_feature_names_out()
coefficients  = model.coef_[0]
top_spam_idx  = np.argsort(coefficients)[-10:][::-1]

print()
print("Top 10 Words Indicating Spam")
print("-" * 35)
for idx in top_spam_idx:
    print(f"  {feature_names[idx]:<20}  weight: {coefficients[idx]:.4f}")


# ── Predict on New Emails ─────────────────────────────────────────────
def predict_email(text):
    cleaned    = clean_email(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    confidence = model.predict_proba(vectorized).max()
    label      = "SPAM" if prediction == 1 else "HAM"
    return label, confidence

new_emails = [
    "Claim your free prize now! You have been selected as a winner",
    "Hi John, just checking in about tomorrow's meeting agenda",
    "URGENT: You owe money. Pay immediately to avoid legal action",
    "The report you requested is attached. Let me know if you need changes",
]

print()
print("Predictions on New Emails")
print("=" * 55)
for email in new_emails:
    label, conf = predict_email(email)
    print(f"Email      : {email[:50]}...")
    print(f"Prediction : {label}  (confidence: {conf:.2%})")
    print()

Model Evaluation
Accuracy : 75.00%

              precision    recall  f1-score   support

         Ham       0.67      1.00      0.80         2
        Spam       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4

Confusion Matrix
-------------------------
             Predicted Ham  Predicted Spam
Actual Ham       2              0
Actual Spam      1              1

Top 10 Words Indicating Spam
-----------------------------------
  free                  weight: 0.2346
  no                    weight: 0.1833
  home                  weight: 0.1718
  from home             weight: 0.1718
  claim                 weight: 0.1700
  claim your            weight: 0.1700
  now                   weight: 0.1667
  click                 weight: 0.1650
  card                  weight: 0.1114
  congratulations you   weight: 0.1114

Predictions on New Emails
